In [0]:
%run /Configurations/Init_Scripts

In [0]:
dbutils.widgets.text('ULLogsTable','silverzone.factullogs')
ULLogsTable = dbutils.widgets.get('ULLogsTable')

dbutils.widgets.text('InvoiceDate','')
InvoiceDate = dbutils.widgets.get('InvoiceDate')

dbutils.widgets.text("DateInterval", "1")
DateInterval = dbutils.widgets.get("DateInterval")

dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

#Value should be in Minutes
dbutils.widgets.text("BufferTime", "20")
BufferTime = dbutils.widgets.get("BufferTime")

dbutils.widgets.dropdown("Frequency", "Monthly", ["Daily", "Monthly"])
Frequency = dbutils.widgets.get("Frequency")

#User can enter multiple SoldToAccountId by comma separator
# dbutils.widgets.text("MidwaySoldToAccountId","0059460093")
# MidwaySoldToAccountId = dbutils.widgets.get("MidwaySoldToAccountId")

assertion_errors = []

In [0]:
def runAssert(condition, message, TestCaseName, InvoiceDate, MismatchedData):
    try:
        assert condition, message
        print(f"\n \t TestCaseName: {TestCaseName}; TestCaseStatus: Passed")
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Pre-Validation",
                    "Status" : "Passed"
                    }]
    except AssertionError as e:
        assertion_errors.append(str(e))
        print(f"\n \t TestCaseName:{TestCaseName}; TestCaseStatus: Failed")
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "TestCaseErrorMessage": message,
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Pre-Validation",
                    "MismatchedData":MismatchedData,
                    "Status" : "Failed"
                    }]

## Execute Test Cases

In [0]:
if InvoiceDate == '' and Frequency == 'Monthly':
    InvoiceDate  = (spark.read.table('promotion.dim_invoicecyclemonth')
                          .filter(col('InvoiceDate').between(date_sub(current_date(), lit(DateInterval)),date_add(current_date(), lit(DateInterval))))
                          .select('InvoiceDate').orderBy(desc('InvoiceDate'))
                           .first()[0]
                          )
    
elif Frequency == 'Daily':
    InvoiceDate  = (spark.read.table('promotion.dim_invoicecyclemonth')
                          .filter(current_timestamp().between(col("CalendarStartTimeStamp"),col("CalendarEndTimeStamp")))
                          .select('InvoiceDate').orderBy(desc('InvoiceDate'))
                           .first()[0]
                          )


print(f"InvoiceDate:{InvoiceDate}")
print(f"Frequnecy:{Frequency}")

In [0]:
%run ./Test_Cases 

In [0]:
validateCycleCount(InvoiceDate,"validateCycleCount")
validateTreatmentDates(InvoiceDate,"validateTreatmentDates")
validateExceptionProcessed(InvoiceDate,"validateExceptionProcessed")
validatePerPatientFeeException(InvoiceDate,"validatePerPatientFeeException")
validateByPatientClassification(InvoiceDate,"validateByPatientClassification")
validateComments(InvoiceDate,"validateComments")
validateNoPatientAssociationFee(InvoiceDate,"validateNoPatientAssociationFee")
validatePerPatientFee(InvoiceDate,"validatePerPatientFee")
validatePerCycleFee(InvoiceDate,"validatePerCycleFee")
validateOffboardingInvoiceAmount(InvoiceDate,"validateOffboardingInvoiceAmount") 
validatePerCycleEfficacyFailFee(InvoiceDate,"validatePerCycleEfficacyFailFee")

In [0]:
# Check if there were any Failures in Test Cases
if assertion_errors:
    raise Exception("Failed Test Cases: " + "; ".join(assertion_errors))
print("All Test Cases are passed.")